<a href="https://colab.research.google.com/github/manek1/de_da_traning/blob/feature_ashwathi/Eligibility_Spark_Code_using_Spark_SQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, coalesce, lit, greatest, least
from pyspark.sql.types import TimestampType


# Initialize Spark session
spark = SparkSession.builder.appName("SQL Conversion").getOrCreate()

# Load CSVs as DataFrames
client = spark.read.option("header", True).csv("/content/sample_data/client.csv")
policy = spark.read.option("header", True).csv("/content/sample_data/policy.csv")
clientcontract = spark.read.option("header", True).csv("/content/sample_data/clientcontract.csv")
countryofresidence = spark.read.option("header", True).csv("/content/sample_data/countryofresidence.csv")
membercoverage = spark.read.option("header", True).csv("/content/sample_data/membercoverage.csv")
groupcoverage = spark.read.option("header", True).csv("/content/sample_data/groupcoverage.csv")
legalentity = spark.read.option("header", True).csv("/content/sample_data/legalentity.csv")
lineofbusiness = spark.read.option("header", True).csv("/content/sample_data/lineofbusiness.csv")
eligibility = spark.read.option("header", True).csv("/content/sample_data/eligibility.csv")
member = spark.read.option("header", True).csv("/content/sample_data/member.csv")
customer = spark.read.option("header", True).csv("/content/sample_data/customer.csv")
clientstaff_category = spark.read.option("header", True).csv("/content/sample_data/clientstaff_category.csv")
# Convert date fields to TimestampType
countryofresidence = countryofresidence.withColumn("countryofresidenceenddate", coalesce(col("countryofresidenceenddate"), lit("9999-12-31 00:00:00")).cast(TimestampType()))
countryofresidence = countryofresidence.withColumn("countryofresidencestartdate", coalesce(col("countryofresidencestartdate"), lit("1900-01-01 00:00:00")).cast(TimestampType()))
eligibility = eligibility.withColumn("eligibilityfromdate", col("eligibilityfromdate").cast(TimestampType()))
eligibility = eligibility.withColumn("eligibilitytodate", col("eligibilitytodate").cast(TimestampType()))
# Temporary Views to simplify logic
client.createOrReplaceTempView("client")
policy.createOrReplaceTempView("policy")
clientcontract.createOrReplaceTempView("clientcontract")
countryofresidence.createOrReplaceTempView("countryofresidence")
membercoverage.createOrReplaceTempView("membercoverage")
groupcoverage.createOrReplaceTempView("groupcoverage")
legalentity.createOrReplaceTempView("legalentity")
lineofbusiness.createOrReplaceTempView("lineofbusiness")
eligibility.createOrReplaceTempView("eligibility")
member.createOrReplaceTempView("member")
customer.createOrReplaceTempView("customer")
clientstaff_category.createOrReplaceTempView("clientstaff_category")

# Run SQL logic using Spark SQL
result = spark.sql(
with cli as (
    select client.clientid, client.clientname, client.sourcesystemid, policy.policyid, policy.policyname,
           client.clientcontractid, policy.premiumcurrencycode, client.PricingMethodCode,
           client.PremiumFundingArrangementCode, clientcontractenddate, clientcontractstartdate,
           clientinceptiondate, clientterminationdate, clientbusinesssegmentcode
    from client
    inner join policy on policy.clientid = client.clientid
        and policy.sourcesystemid = client.sourcesystemid
        and client.currentrecordindicator = 'true'
        and policy.currentrecordindicator = 'true'
    left join clientcontract on client.clientcontractid = clientcontract.clientcontractid
        and client.sourcesystemid = clientcontract.sourcesystemid
),
res_country as (
    select distinct countryofresidencecode, customerid,
        coalesce(countryofresidenceenddate, cast('9999-12-31 00:00:00' as timestamp)) as countryofresidenceenddate,
        coalesce(countryofresidencestartdate, cast('1900-01-01 00:00:00' as timestamp)) as countryofresidencestartdate,
        countryofresidenceenddate as cor_end_date,
        sourcesystemid, etlchecksum as cor_checksum
    from countryofresidence
    where currentrecordindicator = 'true'
        and countryofresidencetypecode = 'ASS'
        and not (countryofresidenceenddate is null and countryofresidencestartdate is null)
),
coverage as (
    select b.primarylegalentityid, b.medicalunderwritingcustomertypecode, a.memberid, a.policyid,
           a.planid, a.packageid, a.sourcesystemid, c.legalentityname, d.lineofbusinessname
    from membercoverage a
    inner join groupcoverage b on a.groupcoverageskey = b.groupcoverageskey
        and a.currentrecordindicator = 'true'
        and b.currentrecordindicator = 'true'
        and a.sourcesystemid = b.sourcesystemid
        and a.areaofcovercode = b.areaofcovercode
        and a.relationshiptypecode = b.relationshiptypecode
        and a.policyid = b.policyid
        and a.planid = b.planid
        and a.packageid = b.packageid
        and a.`clientstaff categoryid` = b.`clientstaff categoryid`
    inner join legalentity c on c.legalentityid = b.primarylegalentityid
        and c.currentrecordindicator = 'true'
        and c.sourcesystemid = b.sourcesystemid
    left join lineofbusiness d on c.lineofbusinessid = d.lineofbusinessid
),
elg as (
    select elg.planid, elg.etlchecksum as elg_checksum, elg.areaofcovercode, elg.relationshiptypecode,
           elg.memberid, elg.familyunittypecode, elg.policyid, memb.dateofbirth, memb.customerid,
           elg.premiumloadingpercentage, coverage.primarylegalentityid, coverage.medicalunderwritingcustomertypecode,
           coverage.legalentityname, coverage.lineofbusinessname, coverage.packageid, memb.clientstaff_categoryid,
           memb.clientstaff_categoryname, elg.eligibilityfromdate, elg.eligibilitytodate, elg.sourcesystemid
    from eligibility elg
    inner join (
        select member.memberid, customer.dateofbirth, customer.customerid,
               member.`clientstaff categoryid` as clientstaff_categoryid, staff.`clientstaff categoryname` as clientstaff_categoryname, member.sourcesystemid
        from member
        inner join customer on member.customerid = customer.customerid
            and member.currentrecordindicator = 'true'
            and customer.currentrecordindicator = 'true'
            and member.sourcesystemid = customer.sourcesystemid
        inner join clientstaff_category staff on member.`clientstaff categoryid` = staff.`clientstaff categoryid`
            and staff.currentrecordindicator = 'true'
            and staff.sourcesystemid = member.sourcesystemid
    ) memb on memb.memberid = elg.memberid and memb.sourcesystemid = elg.sourcesystemid
    inner join coverage on elg.memberid = coverage.memberid
        and elg.policyid = coverage.policyid
        and elg.planid = coverage.planid
        and coverage.sourcesystemid = elg.sourcesystemid
)
select elg.customerid, elg.memberid, cli.clientid, cli.clientname, cli.clientcontractstartdate,
       cli.clientcontractenddate, cli.clientinceptiondate, cli.clientterminationdate,
       cli.clientbusinesssegmentcode, cli.PremiumFundingArrangementCode, cli.PricingMethodCode,
       elg.policyid, cli.policyname, cli.premiumcurrencycode, elg.planid, elg.packageid,
       elg.clientstaff_categoryid, elg.clientstaff_categoryname, elg.primarylegalentityid,
       elg.legalentityname, elg.lineofbusinessname, elg.areaofcovercode, elg.relationshiptypecode,
       elg.familyunittypecode, elg.medicalunderwritingcustomertypecode, elg.premiumloadingpercentage,
       elg.eligibilityfromdate, elg.eligibilitytodate, countryofresidencestartdate, countryofresidenceenddate,
       countryofresidencecode,
       greatest(coalesce(countryofresidencestartdate, cast('1900-01-01 00:00:00' as timestamp)), elg.eligibilityfromdate) as record_from_date,
       least(coalesce(countryofresidenceenddate, cast('9999-12-31 00:00:00' as timestamp)), elg.eligibilitytodate) as record_to_date,
       elg.dateofbirth, elg_checksum, cor_checksum
from elg
inner join cli on elg.policyid = cli.policyid and elg.sourcesystemid = cli.sourcesystemid
left join res_country on res_country.customerid = elg.customerid
    and res_country.sourcesystemid = elg.sourcesystemid
    and (
        elg.eligibilityfromdate between countryofresidencestartdate and countryofresidenceenddate or
        countryofresidencestartdate between elg.eligibilityfromdate and elg.eligibilitytodate
    )
where elg.memberid in ('05044007901', '85000435704', '20026979201', '85100917702')
)
